# Customer-Centric Product Strategy & "Golden Basket" Optimization

This notebook implements an advanced analytical pipeline to transition from raw transaction data to persona-specific product recommendations, referred to as the **Golden Basket**. Using **Polars** for high-performance data processing, the workflow is divided into five strategic phases:

### 1. Data Integration & Segmentation Mapping
* **Merging:** We join historical transaction data with pre-calculated customer clusters using lazy evaluation to maintain an optimized execution graph.
* **Integrity Checks:** Validation of cluster distribution and data consistency post-merge.

### 2. Feature Engineering & Multi-Dimensional Metrics
* **Aggregation:** Computation of key performance indicators (KPIs) per product/cluster, including `reorder_rate`, `basket_presence`, and `spend_share`.
* **Market Penetration:** Calculating the relative reach of each product within specific customer segments.

### 3. Strategic Product Characterization (The Diffusion-Fidelity Matrix)
* **Dynamic Thresholding:** Using cluster-specific quantiles to categorize products into five strategic roles:
    * **Core:** High reach, high loyalty (The backbone of the segment).
    * **Niche:** Low reach, high loyalty (Specialized favorites).
    * **Opportunistic:** High reach, low loyalty (Mass appeal/low frequency).
    * **Companion:** Balanced support products.
    * **Filler:** Low-performing items within the segment.

### 4. Persona-Weighted Optimization (The "Golden Basket")
* **Adaptive Scoring:** Applying custom business logic based on Persona profiles (e.g., prioritizing `healthy_score` for "Premium Healths" vs. `price_score` for "Daily Economizers").
* **Size Constraints:** Dynamically determining the optimal basket size based on the average items per order within each cluster.
* **Selection Logic:** A stratified selection process ensuring a balanced mix of "Core," "Companion," and "Niche" products in the final recommendation.

### 5. Final Deliverables & Visual Validation
* **Artifact Generation:** Exporting the final "Golden Basket" in Parquet and CSV formats for downstream recommendation engines.
* **Strategic Visualizations:** * **Radar Charts:** Profiling personas across Loyalty, Health, and Economy axes.
    * **Role Distribution:** Verifying the balance of the optimized assortment across different customer types.

In [ ]:
import polars as pl
import os

# --- SECTION 1 : MERGE CLUSTER + TRANSACTIONS ---

# 1. Loading the mapping (we retrieve the result of the previous clustering)
# We use the parquet file generated in the previous step for speed
cluster_map = pl.read_parquet("../data/processed/customer_segmentation_results.parquet").select(["user_id", "cluster"])
transactions = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet")

# 3. Join Cluster -> Transactions (Inner Join)
# We transform cluster_map into lazy to remain in the Polars optimisation graph.
transactions_clustered = transactions.join(
    cluster_map.lazy(),
    on="user_id",
    how="inner"
)

# 4. Execution and verification (Collect)
# We take a look at the first few lines to validate the merge.
print("--- Segmented transaction overview ---")
display(transactions_clustered.head().collect())

# 5. Integrity check
# Are all our clusters represented?
check_clusters = transactions_clustered.group_by("cluster").agg(pl.len().alias("nb_lignes")).collect()
print("\n--- Distribution of lines by cluster ---")
print(check_clusters)

transactions_clustered = transactions.join(
    cluster_map.lazy(),
    on="user_id",
    how="inner"
)

# 2. CRITICISM: The result is written to the disk in Parquet format.
# This allows Section 2 (and subsequent sections) to be ultra-fast.
output_path_s1 = "../data/processed/transactions_clustered.parquet"
os.makedirs(os.path.dirname(output_path_s1), exist_ok=True)

# Use .sink_parquet() for large volumes (streaming to disk)
# or .collect().write_parquet() if RAM allows it
transactions_clustered.collect().write_parquet(output_path_s1)

print(f"✅ Section 1 completed: File created here -> {output_path_s1}")

In [ ]:
# --- SECTION 2  ---

# 1. Loading clustered transactions (Lazy)
transactions_clustered = pl.scan_parquet("../data/processed/transactions_clustered.parquet")

# 2. Basic aggregation: Cluster + Product
# We calculate volume, loyalty and value metrics.
product_cluster_stats = (
    transactions_clustered
    .group_by(["cluster", "product_id"])
    .agg([
        pl.col("user_id").n_unique().alias("users"),
        pl.col("order_id").n_unique().alias("baskets"),
        pl.len().alias("orders"),
        pl.col("reordered").mean().alias("reorder_rate"),
        pl.col("order_value").sum().alias("spend"),
        pl.col("healthy").mean().alias("healthy_score"),
        pl.col("cheap_product").mean().alias("price_score")
    ])
)

# 3. Calculating totals by cluster for normalisation
# We need the total number of baskets and total turnover per cluster.
cluster_totals = (
    transactions_clustered
    .group_by("cluster")
    .agg([
        pl.col("order_id").n_unique().alias("cluster_total_baskets"),
        pl.col("order_value").sum().alias("cluster_total_spend"),
        pl.col("user_id").n_unique().alias("cluster_total_users")
    ])
)

#4. Enrichment: Calculation of basket_presence, spend_share, and penetration
product_cluster_stats = (
    product_cluster_stats
    .join(cluster_totals, on="cluster")
    .with_columns([
        # Probability of presence in a basket in the segment
        (pl.col("baskets") / pl.col("cluster_total_baskets")).alias("basket_presence"),

        # Market share of the product within the segment
        (pl.col("spend") / pl.col("cluster_total_spend")).alias("spend_share"),

        # Customer penetration (Relative reach for the cluster)
        (pl.col("users") / pl.col("cluster_total_users")).alias("penetration")
    ])
)

# 5. Execution and Intermediate Saving
# We use collect() because it is the basis for the next classification steps.
df_stats_final = product_cluster_stats.collect()

print("--- Overview of Product x Cluster Statistics ---")
display(df_stats_final.sort(["cluster", "basket_presence"], descending=True).head(10))

# Export for the next section
df_stats_final.write_parquet("../data/processed/product_cluster_stats.parquet")

In [ ]:
# --- SECTION 3  ---
product_cluster_stats = pl.scan_parquet("../data/processed/product_cluster_stats.parquet")

# 2. Calculation of dynamic thresholds (quantiles) PER CLUSTER
# We define what is ‘high’ or ‘low’ within each customer universe.
quantiles = (
    product_cluster_stats
    .group_by("cluster")
    .agg([
        pl.col("basket_presence").quantile(0.25).alias("reach_q25"),
        pl.col("basket_presence").quantile(0.75).alias("reach_q75"),
        pl.col("reorder_rate").quantile(0.25).alias("reorder_q25"),
        pl.col("reorder_rate").quantile(0.75).alias("reorder_q75")
    ])
)

# 3. Joining and Role Assignment
# We apply the logic of 2D space Diffusion vs. Fidelity
pcs = (
    product_cluster_stats
    .join(quantiles, on="cluster")
    .with_columns(
        role = pl.when(
            (pl.col("basket_presence") >= pl.col("reach_q75")) &
            (pl.col("reorder_rate") >= pl.col("reorder_q75"))
        ).then(pl.lit("core"))

        .when(
            (pl.col("basket_presence") >= pl.col("reach_q75")) &
            (pl.col("reorder_rate") < pl.col("reorder_q25"))
        ).then(pl.lit("opportunistic"))

        .when(
            (pl.col("basket_presence") < pl.col("reach_q25")) &
            (pl.col("reorder_rate") >= pl.col("reorder_q75"))
        ).then(pl.lit("niche"))

        .when(
            (pl.col("basket_presence") < pl.col("reach_q25")) &
            (pl.col("reorder_rate") < pl.col("reorder_q25"))
        ).then(pl.lit("filler"))

        .otherwise(pl.lit("companion"))
    )
)

#4. Cleaning: Only the essential columns are kept for the Golden Basket.
final_roles = pcs.select([
    "cluster", "product_id", "role",
    "basket_presence", "reorder_rate", "penetration", "spend_share"
])
df_roles = final_roles.collect()
df_roles.write_parquet("../data/processed/product_role_by_cluster.parquet")

# --- VERIFIYING ---

print("--- Role distribution by cluster ---")
distribution = df_roles.group_by(["cluster", "role"]).len().sort(["cluster", "role"])
print(distribution)

# Quick view of Cluster 0 Top Core (Premium)
print("\n--- Top 5 CORE Products - Cluster 0 (Premium) ---")
display(df_roles.filter((pl.col("cluster") == 0) & (pl.col("role") == "core"))
        .sort("basket_presence", descending=True).head(5))

In [ ]:
# --- SECTION 4: GENERATING THE CUSTOMISED GOLDEN BASKET ---

# 1. DATA PREPARATION & ADAPTIVE SCORES
pcs_lazy = pl.scan_parquet("../data/processed/product_role_by_cluster.parquet")
stats_lazy = pl.scan_parquet("../data/processed/product_cluster_stats.parquet").select([
    "cluster", "product_id", "healthy_score", "price_score"
])
# We retrieve the names of personas to apply business rules.
df_personas = pl.read_csv("../data/processed/df_personas_for_goldenbasket.csv").select([
    pl.col("cluster_id").alias("cluster"),
    "persona"
])

# Initial join
pcs_lazy = pcs_lazy.join(stats_lazy, on=["cluster", "product_id"]).join(df_personas.lazy(), on="cluster")

# Application of BUSINESS LOGIC by Persona
# We define the weights: (Price, Health, Popularity)
pcs_lazy = pcs_lazy.with_columns(
    gb_score = pl.when(pl.col("persona") == "The Premium Healths")
    .then(0.1 * pl.col("price_score") + 0.7 * pl.col("healthy_score") + 0.2 * pl.col("basket_presence"))

    .when(pl.col("persona") == "The Daily Economizers")
    .then(0.7 * pl.col("price_score") + 0.1 * pl.col("healthy_score") + 0.2 * pl.col("basket_presence"))

    .when(pl.col("persona") == "The Budget-Healthy Mix")
    .then(0.4 * pl.col("price_score") + 0.4 * pl.col("healthy_score") + 0.2 * pl.col("basket_presence"))

    .otherwise(0.5 * pl.col("price_score") + 0.5 * pl.col("basket_presence"))
)

pcs_df = pcs_lazy.collect()

# 2. CALCULATION OF SIZE CONSTRAINTS
transactions_clustered = pl.scan_parquet("../data/processed/transactions_clustered.parquet")
cluster_constraints = (
    transactions_clustered
    .group_by("cluster")
    .agg([
        pl.col("order_id").n_unique().alias("total_baskets"),
        pl.len().alias("total_items")
    ])
    .with_columns(
        limit_size = ((pl.col("total_items") / pl.col("total_baskets")) + 1).round(0).cast(pl.Int32)
    )
).collect()

# 3. DYNAMIC SELECTION LOGIC
golden_basket_list = []
for c_info in cluster_constraints.to_dicts():
    c_id = c_info['cluster']
    c_limit = c_info['limit_size']

    # Quotas par rôles
    n_core = max(2, int(c_limit * 0.4))
    n_comp = max(2, int(c_limit * 0.3))
    n_niche = max(1, int(c_limit * 0.2))
    n_opp = 1

    for role_name, n_limit in zip(["core", "companion", "niche", "opportunistic"], [n_core, n_comp, n_niche, n_opp]):
        filtered = (
            pcs_df.filter((pl.col("cluster") == c_id) & (pl.col("role") == role_name))
            .sort("gb_score", descending=True)
            .head(n_limit)
        )
        golden_basket_list.append(filtered)

# 4. FINAL ASSEMBLY & ENRICHMENT
golden_basket = pl.concat(golden_basket_list)

golden_basket_final = (
    golden_basket.lazy()
    .with_columns(
        rank_in_golden = pl.col("gb_score").rank(descending=True).over("cluster")
    )
).collect()

# 5. EXPORT
golden_basket_final.write_parquet("../data/processed/golden_basket_segmented.parquet")

print("✅ Optimised Section 4: Differentiated scoring by customer profile applied.")

In [ ]:
# --- VALIDATION 1 : Profils Healthy/Price ---
validation_persona = golden_basket_final.group_by("cluster").agg([
    pl.col("healthy_score").mean().alias("avg_healthy_in_GB"),
    pl.col("price_score").mean().alias("avg_price_sensitivity_in_GB")
])
print("--- Personas Validation (UK Averages) ---")
print(validation_persona)

# --- VALIDATION 2 : Roles diversity ---
diversity_check = golden_basket_final.group_by(["cluster", "role"]).len().sort("cluster")
print("\n--- Role diversity per basket ---")
print(diversity_check)

In [ ]:
# --- SECTION 5 ---
# We retrieve the Golden Basket already calculated and scored in Section 4.
df_gb_core = pl.scan_parquet("../data/processed/golden_basket_segmented.parquet")

# Persona Layer: Clear Labels
df_personas = pl.read_csv("../data/processed/df_personas_for_goldenbasket.csv").select(["cluster_id", "persona"])

# Metadata layer: Names and categories
df_meta = pl.scan_parquet("../data/processed/df_final_for_pipeline.parquet").select([
    "product_id", "product_name", "aisle", "department"
]).unique()

# --- 2. FINAL ASSEMBLY (JOINTS) ---

golden_basket_final = (
    df_gb_core
    .join(df_personas.lazy(), left_on="cluster", right_on="cluster_id")
    .join(df_meta, on="product_id")
    .select([
        "cluster",
        "persona",
        "product_id",
        "product_name",
        "role",
        "gb_score",
        "rank_in_golden",
        "aisle",
        "department",
        "basket_presence",
        "reorder_rate",
        "spend_share"
    ])
    .sort(["cluster", "rank_in_golden"])
)

# --- 3. EXPORT OF STRATEGIC DELIVERABLES ---

output_dir = "../data/outputs/"
os.makedirs(output_dir, exist_ok=True)

# Deliverable 1: The Golden Basket Artefact (The Holy Grail for recommendation)
final_artifact = golden_basket_final.collect()
final_artifact.write_parquet(f"{output_dir}golden_basket_segmented_final.parquet")
final_artifact.write_csv(f"{output_dir}golden_basket_segmented_final.csv")

# Deliverable 2: Complete mapping of roles (for merchandising)
# We collect the roles of all products, not just the top 8.
pl.read_parquet("../data/processed/product_role_by_cluster.parquet").write_parquet(f"{output_dir}product_role_mapping.parquet")

print(f"🚀 Final artefacts ready in {output_dir}")
print(f"📦 Golden Basket size : {final_artifact.shape[0]} recommendations.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df_viz = final_artifact.to_pandas()

# 2. PREPARATION: Calculation of relative shares (%)
# We group by Persona and Role to see the basket structure.
role_counts = df_viz.groupby(['persona', 'role']).size().unstack().fillna(0)
role_perc = role_counts.div(role_counts.sum(axis=1), axis=0) * 100

# 3. VISUAL CONFIGURATION
plt.figure(figsize=(14, 8))
sns.set_style("white")

colors_map = {
    'core': '#27ae60',
    'companion': '#2980b9',
    'niche': '#8e44ad',
    'opportunistic': '#d35400',
    'filler': '#95a5a6'
}

ordered_roles = [r for r in ['core', 'companion', 'niche', 'opportunistic', 'filler'] if r in role_perc.columns]
role_perc = role_perc[ordered_roles]

# 4. PLOT
ax = role_perc.plot(kind='bar', stacked=True,
                    color=[colors_map[c] for c in ordered_roles],
                    ax=plt.gca(), width=0.7, edgecolor='white', linewidth=1)

#5. ANNOTATIONS (Percentages in the centre of segments)
for p in ax.patches:
    width, height = p.get_width(), p.get_height()
    if height > 5:
        ax.text(p.get_x() + width/2, p.get_y() + height/2, f'{height:.0f}%',
                ha='center', va='center', color='white', fontweight='bold', fontsize=10)

# Displaying the total number of products above each bar
totals = role_counts.sum(axis=1)
for i, total in enumerate(totals):
    ax.text(i, 102, f"Total: {int(total)} products",
            ha='center', fontweight='bold', color='#2c3e50', fontsize=11)

# 6. FINAL DRESSING
plt.title("Strategic Signing of Golden Baskets by Persona\n",
          fontsize=18, fontweight='bold', loc='left', color='#2c3e50')
plt.suptitle("Verification of the balance of the optimised assortment",
             fontsize=12, x=0.25, y=0.92, color='#7f8c8d')

plt.ylabel("Share of Assortment (%)", fontsize=12, fontweight='bold')
plt.xlabel("")
plt.xticks(rotation=0, fontsize=11, fontweight='bold')
plt.ylim(0, 115)

# Légende élégante
plt.legend(title="Product Roles", bbox_to_anchor=(1.02, 1), loc='upper left', frameon=False, fontsize=10)

sns.despine(left=True)
plt.tight_layout()
plt.show()

In [ ]:
import polars as pl
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# 1. Preparation of raw metrics by cluster
# We calculate the actual averages for each desired axis
radar_metrics = (
    pl.scan_parquet("../data/processed/transactions_clustered.parquet")
    .group_by("cluster")
    .agg([
        pl.col("reordered").mean().alias("Loyalty"),
        pl.col("healthy").mean().alias("Health"),
        pl.col("cheap_product").mean().alias("Economy"),
        pl.col("order_value").mean().alias("Basket Value"),
        pl.col("product_id").count().over("order_id").mean().alias("Basket Size")
    ])
    .collect()
)

#2. Adding Persona Names
df_personas = pl.read_csv("../data/processed/df_personas_for_goldenbasket.csv").select(["cluster_id", "persona"])
radar_full = radar_metrics.join(df_personas, left_on="cluster", right_on="cluster_id")

#3. Normalization (Min-Max) for visual rendering
df_radar_raw = radar_full.to_pandas()
categories = [c for c in df_radar_raw.columns if c not in ["cluster", "persona"]]

for col in categories:
    col_min = df_radar_raw[col].min()
    col_max = df_radar_raw[col].max()

    if col_max - col_min != 0:
        df_radar_raw[col] = 0.1 + (df_radar_raw[col] - col_min) / (col_max - col_min) * 0.9
    else:
        df_radar_raw[col] = 0.5
df_radar = df_radar_raw[["persona"] + categories]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

categories = list(df_radar)[1:]
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

n_personas = len(df_radar)
fig, axes = plt.subplots(1, n_personas, figsize=(6 * n_personas, 6), subplot_kw=dict(polar=True))

colors = ['#2ecc71', '#e74c3c', '#3498db', '#f1c40f', '#9b59b6']

for i, persona in enumerate(df_radar['persona']):
    ax = axes[i] if n_personas > 1 else axes
    values = df_radar.iloc[i].drop('persona').values.flatten().tolist()
    values += values[:1]

    # Silhouette
    ax.plot(angles, values, color=colors[i % len(colors)], linewidth=2)
    ax.fill(angles, values, color=colors[i % len(colors)], alpha=0.4)

    # Gray shadow (Overall average)
    mean_values = df_radar.drop('persona', axis=1).mean().values.flatten().tolist()
    mean_values += mean_values[:1]
    ax.plot(angles, mean_values, color='grey', alpha=0.2, linestyle='--')

    ax.set_title(f"Profil : {persona}", size=16, color=colors[i % len(colors)], fontweight='bold', pad=20)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(categories, size=11, fontweight='bold')
    ax.set_yticklabels([])
    ax.set_ylim(0, 1.1)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

df_roles = pd.read_parquet("../data/outputs/product_role_mapping.parquet")
df_personas = pd.read_csv("../data/outputs/persona_profiling.csv")
df_full = df_roles.merge(df_personas, left_on="cluster", right_on="cluster_id")
heatmap_abs = df_full.groupby(['role', 'persona']).size().unstack(fill_value=0)
heatmap_rel = heatmap_abs.div(heatmap_abs.sum(axis=0), axis=1) * 100

# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

# A. Heatmap of gross volumes
sns.heatmap(heatmap_abs, annot=True, fmt="d", cmap="YlGnBu", ax=ax1, cbar_kws={'label': 'Number of products'})
ax1.set_title("Assortment Volume by Role", fontsize=14, fontweight='bold')
ax1.set_xlabel("Persona")
ax1.set_ylabel("Role")

# B. Heatmap of relative densities (%)
sns.heatmap(heatmap_rel, annot=True, fmt=".1f", cmap="magma", ax=ax2, cbar_kws={'label': '% Percentage of the assortment'})
ax2.set_title("Strategic Signature (% of assortment)", fontsize=14, fontweight='bold')
ax2.set_xlabel("Persona")
ax2.set_ylabel("")

plt.suptitle("Comparative Analysis of Catalog Structure by Persona", fontsize=18, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()